In [1]:
%%writefile app.py

import streamlit as st
from pipeline import generate_highlight, clear_temp_folder
import os


st.set_page_config(
    page_title="YouTube Highlight Generator",
    page_icon="🎬",
    layout="wide"
)


st.title("🎬 YouTube Highlight Generator")


st.write(
    """
Paste a YouTube URL.

The application will

- Download the video
- Convert speech to text
- Detect important moments
- Create a 1 minute highlight video
"""
)


url = st.text_input(
    "YouTube URL",
    placeholder="https://www.youtube.com/watch?v=xxxxxxxx"
)


# Optional manual cleanup button
if st.button("Clear Previous Files"):

    clear_temp_folder()

    st.success(
        "Previous files cleared."
    )


if st.button("Generate Highlight"):

    if url == "":

        st.error(
            "Please enter a YouTube URL."
        )

        st.stop()


    progress = st.progress(0)

    status = st.empty()

    try:

        output_video = generate_highlight(
            url=url,
            progress=progress,
            status=status
        )


        st.success(
            "Highlight generated!"
        )


        # unique key prevents old video cache
        st.video(
            output_video
        )


        with open(
            output_video,
            "rb"
        ) as file:


            st.download_button(

                "Download Highlight",

                file,

                file_name="highlight_video.mp4",

                mime="video/mp4",

                key=output_video

            )


    except Exception as e:

        st.error(e)

Writing app.py


In [2]:
%%writefile pipeline.py
import os
import json
import re
import subprocess
import yt_dlp
import whisper
import contractions
import streamlit as st
import torch
import shutil
import time

from transformers import pipeline

from transformers import pipeline
from moviepy.editor import (
    VideoFileClip,
    concatenate_videoclips
)

TEMP_DIR = "temp"

os.makedirs(TEMP_DIR, exist_ok=True)

def clear_temp_folder():

    if os.path.exists(TEMP_DIR):

        for item in os.listdir(TEMP_DIR):

            path = os.path.join(
                TEMP_DIR,
                item
            )

            try:

                if os.path.isfile(path):

                    os.remove(path)


                elif os.path.isdir(path):

                    shutil.rmtree(path)


            except Exception as e:

                print(
                    "Cleanup error:",
                    e
                )

    else:

        os.makedirs(
            TEMP_DIR,
            exist_ok=True
        )

# load models
@st.cache_resource
def load_whisper_model():

    print("Loading Whisper model...")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = whisper.load_model(
        "base",
        device=device
    )

    print("Whisper model loaded")

    return model



@st.cache_resource
def load_classifier():

    device = 0 if torch.cuda.is_available() else -1

    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=device
    )



    print("Classification model loaded")
    return classifier


FILLERS = [
    "um",
    "uh",
    "erm",
    "ah",
    "hmm",
    "you know",
    "like",
    "sort of",
    "kind of"
]


def clean_text(text):

    text = contractions.fix(text)

    text = text.lower()

    for filler in FILLERS:

        pattern = r"\b" + re.escape(filler) + r"\b"

        text = re.sub(pattern, "", text)

    text = re.sub(r"[^\w\s?.!]", "", text)

    text = re.sub(r"\?+", "?", text)

    text = re.sub(r"!+", "!", text)

    text = re.sub(r"\.+", ".", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def generate_highlight(url, progress, status):

    # remove previous generated files
    clear_temp_folder()

    # create unique ID for this run
    run_id = str(int(time.time()))

    ###############################################
    # Download video
    ###############################################

    status.text("Downloading video...")
    progress.progress(10)

    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(
            TEMP_DIR,
            f"video_{run_id}.%(ext)s"
        ),
        "quiet": False
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

        # Get the actual downloaded filename
        video_path = ydl.prepare_filename(info)

        # If merged to mp4, update the extension
        if video_path.endswith(".webm") or video_path.endswith(".mkv"):
            mp4_path = os.path.splitext(video_path)[0] + ".mp4"
            if os.path.exists(mp4_path):
                video_path = mp4_path

    print("Downloaded video:", video_path)
    print("Exists:", os.path.exists(video_path))
    print("Size:", os.path.getsize(video_path))

    ###############################################
    # Extract Audio
    ###############################################

    status.text("Extracting audio...")

    progress.progress(20)

    audio_path = os.path.join(TEMP_DIR, "audio.wav")

    subprocess.run([

        "ffmpeg",

        "-i",

        video_path,

        "-ar",

        "16000",

        "-ac",

        "1",

        audio_path,

        "-y"

    ])

    ###############################################
    # Load Whisper
    ###############################################

    status.text("Loading Whisper model...")

    progress.progress(30)

    model = load_whisper_model()

    ###############################################
    # Transcribe
    ###############################################

    status.text("Transcribing audio...")

    progress.progress(40)

    result = model.transcribe(

        audio_path,

        word_timestamps=True

    )

    ###############################################
    # Save transcript
    ###############################################

    transcript=[]

    for seg in result["segments"]:

        transcript.append({

            "start":seg["start"],

            "end":seg["end"],

            "text":seg["text"]

        })

    with open(

        os.path.join(TEMP_DIR,"transcript.json"),

        "w",

        encoding="utf8"

    ) as f:

        json.dump(transcript,f,indent=4)

    ###############################################
    # Clean transcript
    ###############################################

    status.text("Cleaning transcript...")

    progress.progress(50)

    cleaned_segments=[]

    for segment in transcript:

        cleaned_segments.append({

            "start":segment["start"],

            "end":segment["end"],

            "text":clean_text(segment["text"])

        })

    with open(

        os.path.join(TEMP_DIR,"cleaned.json"),

        "w",

        encoding="utf8"

    ) as f:

        json.dump(cleaned_segments,f,indent=4)

    ###############################################
    # Zero-shot classification
    ###############################################

    status.text("Loading classification model...")

    progress.progress(60)


    classifier = load_classifier()


    labels = [
        "Question",
        "Answer",
        "Agreement",
        "Disagreement",
        "Neutral"
    ]


    valid_segments = [

        seg for seg in cleaned_segments

        if seg["text"].strip()

    ]


    events = []


    total = len(valid_segments)


    for i, seg in enumerate(valid_segments):


        result = classifier(

            seg["text"],

            candidate_labels=labels

        )


        label = result["labels"][0]


        confidence = float(
            result["scores"][0]
        )


        events.append({

            "start": seg["start"],

            "end": seg["end"],

            "text": seg["text"],

            "label": label,

            "confidence": round(
                confidence,
                3
            )

        })


        # update progress

        if i % 10 == 0:

            percent = 60 + int(
                (i / total) * 15
            )

            progress.progress(percent)


    classified_path = os.path.join(
        TEMP_DIR,
        "classified_segments.json"
    )


    with open(
        classified_path,
        "w",
        encoding="utf8"
    ) as f:

        json.dump(
            events,
            f,
            indent=4
        )


    ###############################################
    # Event Detection
    ###############################################

    status.text("Detecting important events...")

    progress.progress(75)


    classified_segments = events


    detected_events = []


    CONFIDENCE_THRESHOLD = 0.45



    for i, seg in enumerate(classified_segments):


        label = seg["label"]

        confidence = seg["confidence"]



        if confidence < CONFIDENCE_THRESHOLD:

            continue



        #################################
        # Question Answer Detection
        #################################

        if label == "Question":


            answer = None


            for j in range(

                i + 1,

                min(
                    i + 4,
                    len(classified_segments)
                )

            ):


                next_seg = classified_segments[j]


                if (

                    next_seg["label"] == "Answer"

                    and

                    next_seg["confidence"]

                    >= CONFIDENCE_THRESHOLD

                ):


                    answer = next_seg

                    break



            if answer:


                detected_events.append({

                    "event":
                    "Question-Answer",


                    "start":
                    seg["start"],


                    "end":
                    answer["end"],


                    "question":
                    seg["text"],


                    "answer":
                    answer["text"],


                    "confidence":

                    round(

                        (

                            seg["confidence"]

                            +

                            answer["confidence"]

                        )

                        / 2,

                        3

                    )

                })



        #################################
        # Agreement
        #################################

        elif label == "Agreement":


            detected_events.append({

                "event":
                "Agreement",


                "start":
                seg["start"],


                "end":
                seg["end"],


                "text":
                seg["text"],


                "confidence":
                confidence

            })



        #################################
        # Disagreement
        #################################

        elif label == "Disagreement":


            detected_events.append({

                "event":
                "Disagreement",


                "start":
                seg["start"],


                "end":
                seg["end"],


                "text":
                seg["text"],


                "confidence":
                confidence

            })



    detected_path = os.path.join(

        TEMP_DIR,

        "detected_events.json"

    )


    with open(

        detected_path,

        "w",

        encoding="utf8"

    ) as f:


        json.dump(

            detected_events,

            f,

            indent=4

        )


    print(
        "Detected events:",
        len(detected_events)
    )


    ###############################################
    # Create Highlight Video
    ###############################################

    status.text("Creating highlight video...")

    progress.progress(90)

    status.text(" Loading video path ")
    VIDEO_PATH = video_path

    EVENTS_PATH = detected_path


    BUFFER_BEFORE = 5

    BUFFER_AFTER = 5

    MAX_CLIP_DURATION = 10

    TARGET_DURATION = 60


    status.text(" Loading video clip ")
    video = VideoFileClip(
        VIDEO_PATH
    )


    video_duration = video.duration



    with open(
        EVENTS_PATH,
        "r",
        encoding="utf8"
    ) as f:

        events = json.load(f)



    # Sort important events first

    events = sorted(

        events,

        key=lambda x:

        x["confidence"],

        reverse=True

    )



    selected_clips = []

    selected_events = []

    total_duration = 0



    ###############################################
    # Select best clips
    ###############################################
    status.text(" Selecting best clips ")
    for event in events:


        if total_duration >= TARGET_DURATION:

            break



        start = max(

            0,

            event["start"] - BUFFER_BEFORE

        )


        end = min(

            video_duration,

            event["end"] + BUFFER_AFTER

        )



        duration = end - start



        if duration > MAX_CLIP_DURATION:


            end = start + MAX_CLIP_DURATION


            duration = MAX_CLIP_DURATION



        # Avoid overlapping clips

        overlap = False


        for selected in selected_events:


            if abs(

                start -

                selected["start"]

            ) < 15:


                overlap = True

                break



        if overlap:

            continue



        selected_events.append({

            "start": start,

            "end": end,

            "event": event["event"]

        })



        clip = video.subclip(

            start,

            end

        )


        selected_clips.append(clip)



        total_duration += duration



    print(
        "Selected clips:",
        len(selected_clips)
    )



    ###############################################
    # Combine clips
    ###############################################
    status.text(" combining clips ")
    output_path = os.path.join(
        TEMP_DIR,
        f"highlight_video_{run_id}.mp4"
    )



    if selected_clips:


        final_video = concatenate_videoclips(

            selected_clips,

        )



        # Limit to 60 seconds

        if final_video.duration > TARGET_DURATION:


            final_video = final_video.subclip(

                0,

                TARGET_DURATION

            )


        status.text(" final video ")
        final_video = final_video.resize(width=1280)

        final_video.write_videofile(
            output_path,
            codec="libx264",
            audio_codec="aac",
            preset="ultrafast"
        )



        final_video.close()



    else:


        raise Exception(
            "No important events detected."
        )



    video.close()



    progress.progress(100)


    status.text(
        "Completed!"
    )


    return output_path



Writing pipeline.py


In [4]:
%%writefile requirements.txt
streamlit
yt-dlp
openai-whisper
torch
transformers
moviepy==1.0.3
contractions
sentencepiece
accelerate

Writing requirements.txt


In [5]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 31.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=1f87e39fa3d2c9161fe8adbb1847bec4a5c5a026cd11d0dd7e16cd19677ff43f
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-

In [6]:
!apt-get update -qq
!apt-get install ffmpeg -y -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [7]:
!pip install pyngrok streamlit

In [8]:
from pyngrok import ngrok

ngrok.set_auth_token("$TOKEN")

In [9]:
!ls

app.py	pipeline.py  requirements.txt  sample_data


In [10]:
import subprocess

subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.headless",
        "true"
    ]
)

<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>

In [11]:
ngrok.kill()
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://residue-dancing-veggie.ngrok-free.dev" -> "http://localhost:8501"
